# 🌊 Day 05: Kadane's Algorithm & Subarray Patterns

---

## The Most Elegant DP You'll Ever See

Kadane's algorithm is one of those things that seems too simple to be this powerful. It solves Maximum Subarray in O(n) with a single pass.

**The core idea:** At each position, make a decision — do I extend the previous subarray, or start fresh from here?

```
nums:        [-2,  1, -3,  4, -1,  2,  1, -5,  4]
             
At each i:   Should I include the previous subarray sum?
             Or is it dragging me down (negative sum)?
             
Decision:    current = max(nums[i], current + nums[i])
```

That's it. That's Kadane's.

---

## 🎯 Problem 1: Maximum Subarray (LC #53)

The classic. If you only learn one DP algorithm, make it this one.

```
Find the contiguous subarray with the largest sum.
Input: [-2, 1, -3, 4, -1, 2, 1, -5, 4]  →  Output: 6
(subarray [4, -1, 2, 1] has the largest sum = 6)
```

In [ ]:
def maxSubArray(nums):
    """
    Kadane's Algorithm
    Time: O(n), Space: O(1)
    """
    current_sum = max_sum = nums[0]
    
    for num in nums[1:]:
        # Either extend previous subarray, or start fresh
        current_sum = max(num, current_sum + num)
        max_sum = max(max_sum, current_sum)
    
    return max_sum

# Let's trace it step by step
nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
print(f"Max subarray sum: {maxSubArray(nums)}")  # 6

# Trace:
current = nums[0]
print(f"Start: current={current}")
for i, num in enumerate(nums[1:], 1):
    old = current
    current = max(num, current + num)
    decision = "EXTEND" if current == old + num else "RESTART"
    print(f"  i={i}, num={num:+d}, current={current:+d}  [{decision}]")

---

## 🎯 Problem 2: Maximum Product Subarray (LC #152)

Kadane's evil twin. Same idea, but with multiplication — and negatives flip everything.

```
Input: [2, 3, -2, 4]  →  Output: 6 (subarray [2, 3])
Input: [-2, 0, -1]    →  Output: 0
```

**The twist:** A negative × negative = positive. So the smallest (most negative) product could suddenly become the largest. Track BOTH min and max at each step.

In [ ]:
def maxProduct(nums):
    """
    Modified Kadane's: track both min and max products.
    Time: O(n), Space: O(1)
    """
    result = cur_max = cur_min = nums[0]
    
    for num in nums[1:]:
        # If num is negative, max and min swap roles!
        if num < 0:
            cur_max, cur_min = cur_min, cur_max
        
        cur_max = max(num, cur_max * num)
        cur_min = min(num, cur_min * num)
        result = max(result, cur_max)
    
    return result

print(maxProduct([2, 3, -2, 4]))    # 6
print(maxProduct([-2, 0, -1]))       # 0
print(maxProduct([-2, 3, -4]))       # 24 (entire array!)

---

## 🎯 Problem 3: Best Time to Buy and Sell Stock (LC #121)

Technically a Kadane's variant — find max profit, which is the max subarray of daily differences.

But the direct approach is even simpler:

```
Input: [7, 1, 5, 3, 6, 4]  →  Output: 5
(Buy at 1, sell at 6 → profit = 5)
```

In [ ]:
def maxProfit(prices):
    """
    Track minimum price seen so far, maximize profit at each step.
    Time: O(n), Space: O(1)
    """
    min_price = float('inf')
    max_profit = 0
    
    for price in prices:
        min_price = min(min_price, price)
        max_profit = max(max_profit, price - min_price)
    
    return max_profit

print(maxProfit([7, 1, 5, 3, 6, 4]))  # 5
print(maxProfit([7, 6, 4, 3, 1]))      # 0 (always falling)

---

## 🎯 Problem 4: Maximum Sum Circular Subarray (LC #918)

What if the subarray can wrap around? Like `[5, -3, 5]` → the answer is `5 + 5 = 10` (skipping the -3 by wrapping).

**Trick:** The max circular subarray is either:
1. A normal max subarray (Kadane's), OR
2. `total_sum - min_subarray` (everything except the minimum chunk in the middle)

```
[.. max_wrap_left ..][  min_subarray  ][.. max_wrap_right ..]
                      └── subtract this to get the circular max
```

In [ ]:
def maxSubarraySumCircular(nums):
    """
    Max of (normal Kadane's, total - min Kadane's)
    Time: O(n), Space: O(1)
    """
    total = 0
    max_sum = cur_max = nums[0]
    min_sum = cur_min = nums[0]
    
    for num in nums:
        total += num
    
    # Normal Kadane's for max and min
    cur_max = cur_min = 0
    max_sum = float('-inf')
    min_sum = float('inf')
    
    for num in nums:
        cur_max = max(num, cur_max + num)
        max_sum = max(max_sum, cur_max)
        cur_min = min(num, cur_min + num)
        min_sum = min(min_sum, cur_min)
    
    # Edge case: all negative → total - min_sum = 0, but max_sum is correct
    if max_sum < 0:
        return max_sum
    
    return max(max_sum, total - min_sum)

print(maxSubarraySumCircular([5, -3, 5]))       # 10
print(maxSubarraySumCircular([1, -2, 3, -2]))    # 3
print(maxSubarraySumCircular([-3, -2, -3]))      # -2

---

## 🗺️ Subarray Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  SUBARRAY PATTERN VARIANTS                                          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. MAX SUM SUBARRAY: Kadane's → extend or restart                  ║
║     current = max(num, current + num)                                ║
║                                                                      ║
║  2. MAX PRODUCT: Track both min and max (negatives flip!)           ║
║     Swap min/max when num is negative                                ║
║                                                                      ║
║  3. BUY/SELL STOCK: Track min_price, maximize price - min_price     ║
║                                                                      ║
║  4. CIRCULAR: max(kadane_max, total - kadane_min)                   ║
║     Watch for all-negative edge case                                 ║
║                                                                      ║
║  5. SUBARRAY SUM = K: Prefix sum + hashmap (see Day 03)            ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 53 | Maximum Subarray | Medium | Kadane's |
| 152 | Maximum Product Subarray | Medium | Min-Max Kadane's |
| 121 | Best Time to Buy/Sell Stock | Easy | Min tracking |
| 918 | Max Sum Circular Subarray | Medium | Kadane's + total-min |
| 122 | Buy/Sell Stock II (multiple) | Medium | Greedy |
| 978 | Longest Turbulent Subarray | Medium | Modified Kadane's |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Kadane's = at each step, extend or restart: max(num, cur+num)  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Product subarray: track BOTH min and max (negatives flip)      │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Buy/Sell Stock is Kadane's in disguise: track running minimum  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Circular = max(normal_max, total - normal_min)                 │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Any "max/min contiguous subarray" → think Kadane's first      │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Heaps & Top-K Problems

When you need the K largest/smallest/closest — there's a heap for that.

---

### 🎉 Subarray Patterns Done!
Kadane's is your best friend for any contiguous subarray optimization.